# Hyperparameter Sensitivity Analysis — ManiMamba (Weather)

Reads `sensitivity_results.xlsx` and produces sensitivity figures.

In [ ]:
%matplotlib inline

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

XLSX_PATH = "sensitivity_results.xlsx"
OPTUNA_DIR = os.path.join("..", "..", "output", "optuna")

RED = "#FFC000"
SKY = "#3D5497"
GRAY = "#555555"

PARAMS = ["d_model", "epsilon", "geo_d_model", "cov_rank"]
PARAM_TITLES = {
    "d_model": "Model Dimension",
    "epsilon": r"SPD Regularization ($\epsilon$)",
    "geo_d_model": "Geometry Mamba Dim",
    "cov_rank": "Covariance Rank",
}

df = pd.read_excel(XLSX_PATH)
df = df[df["status"].str.startswith(("ok",))].copy()
df["mse"] = pd.to_numeric(df["mse"], errors="coerce")
df["mae"] = pd.to_numeric(df["mae"], errors="coerce")
df = df.dropna(subset=["mse", "mae"])

PRED_LENS = sorted(df["pred_len"].unique())

def load_best_param(pred_len, param):
    path = os.path.join(OPTUNA_DIR, f"ManiMamba_Weather_pl{pred_len}_best_params.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        data = json.load(f)
    return data["best_params"].get(param)

baseline = {}
for pl in PRED_LENS:
    baseline[pl] = {p: load_best_param(pl, p) for p in PARAMS}

print(f"Loaded {len(df)} valid results for pred_lens={PRED_LENS}")
print("Baseline (best) values from Optuna:")
for pl in PRED_LENS:
    print(f"  pl={pl}: {baseline[pl]}")
display(df)

In [ ]:
PL_COLORS = {}
if len(PRED_LENS) >= 2:
    PL_COLORS[PRED_LENS[0]] = (RED, "o", "-")
    PL_COLORS[PRED_LENS[1]] = (SKY, "s", "--")
elif len(PRED_LENS) == 1:
    PL_COLORS[PRED_LENS[0]] = (RED, "o", "-")

fig, axes = plt.subplots(1, len(PARAMS), figsize=(20, 3.2), constrained_layout=True)

for ax, param in zip(axes, PARAMS):
    sub = df[df["sweep_param"] == param].sort_values("sweep_value")
    if sub.empty:
        ax.text(0.5, 0.5, f"No data for\n{param}", ha="center", va="center",
                transform=ax.transAxes, fontsize=10, color=GRAY)
        ax.set_title(PARAM_TITLES.get(param, param), fontsize=10)
        continue

    for pl in PRED_LENS:
        pl_sub = sub[sub["pred_len"] == pl].copy()
        if pl_sub.empty:
            continue
        color, marker, ls = PL_COLORS[pl]

        ax.plot(
            pl_sub["sweep_value"], pl_sub["mse"],
            color=color, marker=marker, linestyle=ls,
            linewidth=1.8, markersize=6, label=f"pl={pl}",
            zorder=3,
        )

    oom_sub = df[(df["sweep_param"] == param) & (df["status"] == "oom")]
    for _, row in oom_sub.iterrows():
        ylim_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1
        ax.scatter(row["sweep_value"], ylim_top,
                   color=GRAY, marker="x", s=50, zorder=4)

    ax.set_title(PARAM_TITLES.get(param, param), fontsize=10, fontweight="bold")
    ax.set_ylabel("MSE", fontsize=9)
    ax.tick_params(labelsize=8)

    if param == "epsilon":
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
        ax.ticklabel_format(axis="x", style="scientific", scilimits=(0, 0))
    elif param in ("d_model", "geo_d_model"):
        ax.set_xscale("log", base=2)
        ax.set_xticks(sub["sweep_value"].unique())
        ax.set_xticklabels([str(int(v)) for v in sub["sweep_value"].unique()])
    else:
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.2, linewidth=0.5)

axes[-1].legend(fontsize=8, loc="upper right", framealpha=0.8)

fig.suptitle("Hyperparameter Sensitivity — ManiMamba on Weather", fontsize=11, y=1.02)
plt.show()

In [ ]:
print("=== Summary Table (MSE) ===")
pivot_mse = df.pivot_table(index="sweep_value", columns=["pred_len", "sweep_param"], values="mse")
display(pivot_mse.round(6))

print("=== Summary Table (MAE) ===")
pivot_mae = df.pivot_table(index="sweep_value", columns=["pred_len", "sweep_param"], values="mae")
display(pivot_mae.round(6))

In [ ]:
paper_params = ["d_model", "epsilon", "geo_d_model", "cov_rank"]
paper_titles = {
    "d_model": "(a) Hidden Dimension",
    "epsilon": "(b) SPD Regularization",
    "geo_d_model": "(c) Geometry Mamba Dim",
    "cov_rank": "(d) Covariance Rank",
}

def _tick_decimals(ax):
    ticks = ax.yaxis.get_major_locator().tick_values(*ax.get_ylim())
    if len(ticks) < 2:
        return 4
    step = min(abs(ticks[i + 1] - ticks[i]) for i in range(len(ticks) - 1))
    if step <= 0:
        return 4
    return min(5, max(2, int(np.ceil(-np.log10(step)))))

fig, axarr = plt.subplots(2, 2, figsize=(5.5, 3.8))
axes = axarr.flatten()

for idx, param in enumerate(paper_params):
    ax_l = axes[idx]
    ax_r = ax_l.twinx()

    sub = df[df["sweep_param"] == param].sort_values("sweep_value")
    sweep_vals = sub["sweep_value"].unique()
    x_pos = np.arange(len(sweep_vals))

    if sub.empty:
        ax_l.text(0.5, 0.5, f"No data for\n{param}",
                 ha="center", va="center", transform=ax_l.transAxes,
                 fontsize=8, color=GRAY)
        ax_l.set_title(paper_titles.get(param, param), fontsize=7.5, fontweight="bold", pad=4)
        ax_r.set_visible(False)
        continue

    if param == "epsilon":
        xtick_labels = [f"{v:.0e}".replace("e+0", "e").replace("e-0", "e-") for v in sweep_vals]
    else:
        xtick_labels = [str(int(v)) for v in sweep_vals]

    for pl_i, (pl, color, marker, ls, ms) in enumerate([
        (PRED_LENS[0], RED, "s", "-", 6),
        (PRED_LENS[1], "#2E75B6", "^", "--", 8),
    ]):
        pl_sub = sub[sub["pred_len"] == pl].copy()
        if pl_sub.empty:
            continue

        val_to_pos = {v: p for p, v in enumerate(sweep_vals)}
        pl_x = np.array([val_to_pos[v] for v in pl_sub["sweep_value"]])

        ax = ax_l if pl_i == 0 else ax_r
        ax.plot(
            pl_x, pl_sub["mse"].values,
            color=color, marker=marker, linestyle=ls,
            linewidth=1.0, markersize=ms, label=f"Pred Len={pl}",
            markeredgecolor="black", markeredgewidth=0.5,
            markerfacecolor=color,
            zorder=3,
        )

    for ax in (ax_l, ax_r):
        ymin, ymax = ax.get_ylim()
        margin = (ymax - ymin) * 0.12
        ax.set_ylim(ymin - margin, ymax + margin)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=False))
        dec = _tick_decimals(ax)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(f"%.{dec}f"))
        ax.tick_params(axis="y", labelcolor="black", labelsize=7, length=2, pad=1)
        for label in ax.get_yticklabels():
            label.set_fontweight("bold")

    ax_l.set_xlim(x_pos[0] - 0.4, x_pos[-1] + 0.4)
    ax_l.set_xticks(x_pos)
    ax_l.set_xticklabels(xtick_labels, fontsize=7)
    ax_l.tick_params(axis="x", labelsize=7, length=2, pad=1)
    for label in ax_l.get_xticklabels():
        label.set_fontweight("bold")
    ax_l.set_title(paper_titles[param], fontsize=7.5, fontweight="bold", pad=4)

    ax_l.spines["top"].set_visible(True)
    ax_r.spines["top"].set_visible(True)
    ax_l.grid(True, alpha=0.3, linewidth=0.4, color="#AAAAAA")

    if idx == 1:
        lines_l, labels_l = ax_l.get_legend_handles_labels()
        lines_r, labels_r = ax_r.get_legend_handles_labels()
        ax_l.legend(lines_l + lines_r, labels_l + labels_r,
                    fontsize=6, loc="lower center", framealpha=0.7,
                    handlelength=1.8, borderpad=0.4, labelspacing=0.3)

fig.text(0.04, 0.5, "MSE", va="center", rotation=90, fontsize=8, fontweight="bold")

fig.tight_layout(pad=0.2, h_pad=0.6, w_pad=1.0)
fig.subplots_adjust(left=0.12, right=0.88)
fig.set_dpi(300)
plt.show()